In [2]:
from loading_utils import load_spike_data, match_times
from analysis_utils import make_correlation_dictionary, visualize_correlation_dictionary

spike_df = load_spike_data()

# Finds which spikes happened during a SWR
df = match_times(
    spike_df,
    "../src/SWRs_7744_partner_intro.csv",
    filter_event_data = {"KSLabel": ["good"]},
    keep_event_columns = ["Time", "Cluster ID"],
    progress=True,
)

print(spike_df.head(5))
print(df.head(5))

Checking Event Data: 100%|██████████| 136/136 [00:00<00:00, 339.84it/s]


       Time  Cluster ID KSLabel
0  0.006367          16    good
1  0.007867          67    good
2  0.010267          16    good
3  0.011467          94    good
4  0.011800         119     mua
      Start      Peak      Stop    Trough  Max_Envelope_Amp  Duration_ms  \
0  395.9536  395.9680  395.9792  395.9680          8.422738         25.6   
1  396.4532  396.4728  396.4908  396.4728          8.061243         37.6   
2  397.1432  397.1540  397.1636  397.1540          6.616578         20.4   
3  397.6024  397.6056  397.6284  397.6056          2.276005         26.0   
4  397.8448  397.8552  397.8728  397.8552         10.587707         28.0   

                                         Event Times  \
0  [395.95636666666667, 395.9578666666667, 395.96...   
1  [396.4537, 396.4540333333333, 396.455766666666...   
2  [397.1432, 397.1439, 397.14636666666667, 397.1...   
3  [397.6025333333333, 397.60273333333333, 397.60...   
4  [397.8448, 397.84516666666667, 397.84543333333...   

              

In [ ]:
# Collects correlation data for neuron clusters
corr_matrix = make_correlation_dictionary(df, normalize=False)

# Plots the data in a 2D correlation matrix
visualize_correlation_dictionary(corr_matrix)

In [ ]:
import sys
import pandas as pd
import numpy as np
from tqdm import tqdm

def match_times(
    dataframe: pd.DataFrame,
    directory: str,
    filter_event_data: dict = {},
    filter_window_data: dict = {},
    event_time_column: str = "Time",
    time_interval_columns: list = ["Start", "Stop"],
    keep_event_columns: list = [],
    progress: bool = True,
) -> pd.DataFrame:

    """
    Checks if events contained in a user-provided DataFrame
    contain times that fall within a window found in a
    separate datafile. For consistency, the user-provided
    DataFrame is referred to as the 'event dataframe' and
    should only contain a single time per row. The directory
    should point towards the 'window dataframe' which should
    contain two times per row (a start and stop time).

    Parameters:
    -----------
    dataframe :: pd.DataFrame
        Dataframe containing at least one column
        labeled 'Time' (or the string assigned to
        'event_time_column'; assumes units of seconds).
        This function will look for time ranges that
        these times fall between.
    directory :: str
        Path (directory + filename) containing data with
        times ranges (found in columns 'Start' and 'Stop'
        by default).
    filter_event_data :: dict
        Indicates which values to filter by in the 'event
        dataframe.' Every key-value pair should be a column
        name in the 'event dataframe' (key) and a list of
        valid entries (value). If no arguments are provided,
        then no filtering is applied.
    filter_window_data :: dict
        Indicates which values to filter by in the 'window
        dataframe.' Every key-value pair should be a column
        name in the 'window dataframe' (key) and a list of
        valid entries (value). If no arguments are provided,
        then no filtering is applied.
    event_time_column :: str
        The name of a column in the 'event dataframe' containing
        time data (assumes units of seconds).
    time_interval_column :: list
        The names of two columns in the 'window dataframe'
        containing the start and stop (both in seconds) of
        a valid window. Events with a time falling between these
        two values will be associated with the respective window.
    keep_event_columns :: list
        A list of columns to keep from the 'event dataframe'. For
        example, if provided ['A', 'B'], then the values of 'A' and
        'B' for all matched events will be saved in a list in the
        returned dataframe.
    progress :: bool
        Disables / enables progress bar.

    Returns:
    --------
    return_df :: pd.DataFrame
        A copy of the initial DataFrame, but with
        two new columns containing the SWR start
        and stop times. If no matching SWR data
        was found, both columns should default
        to NaN values.
    """

    # This prevents us from accidentally modifying the original DataFrame
    try:
        event_df = dataframe.copy()
    except AttributeError:
        print("Could not copy the provided DataFrame")
        sys.exit(1)

    try:
        window_df = pd.read_csv(directory)
    except FileNotFoundError:
        assert FileNotFoundError(f"Filename '{directory}' not found")
        sys.exit(1)

    # NOTE: These could probably be broken off into a separate function, will revisit later
    for column_name, valid_values in filter_event_data.items():
        try:
            event_df = event_df[event_df[column_name].isin(valid_values)]
        except KeyError:
            print(f"Column name '{key}' not found, moving on without filtering")
        except TypeError:
            print(f"Valid values must be a list, not '{type(valid_values)}'")

    for column_name, valid_values in filter_window_data.items():
        try:
            window_df = window_df[window_df[column_name].isin(valid_values)]
        except KeyError:
            print(f"Column name '{key}' not found, moving on without filtering")
        except TypeError:
            print(f"Valid values must be a list, not '{type(valid_values)}'")

    # Renames columns to be grammatically correct
    renamed_event_columns = [f"Event {string}s" for string in keep_event_columns]
    
    # Initialize new columns properly (object dtype) to prevent Pandas errors
    for col in renamed_event_columns:
        window_df[col] = None
        window_df[col] = window_df[col].astype("object")
    
    # Extract all relevant time data (units of seconds)
    event_times = np.array(event_df[event_time_column])
    
    for idx in tqdm(range(len(window_df)), desc="Checking Event Data", disable=not progress):

        # Masking allows us to avoid excessive looping
        start_time = np.array(window_df[time_interval_columns[0]])[idx]
        end_time = np.array(window_df[time_interval_columns[1]])[idx]
        mask = (start_time <= event_times) & (event_times <= end_time)

        # NOTE: Apparently Pandas is depricating their old indexing tricks, need to use '.at' now
        for column, renamed_column in zip(keep_event_columns, renamed_event_columns):
            data = list(event_df[column][mask])
            window_df.at[idx, renamed_column] = data

    return window_df

spike_df = load_spike_data()

# df = match_times(
#     spike_df,
#     "../src/SWRs_7744_partner_intro.csv",
#     filter_event_data = {"KSLabel": ["good"]},
#     keep_event_columns = ["Time", "Cluster ID"],
#     progress=True,
# )

# print(df.head(5))

df = match_times(
    spike_df,
    "../test_data/test_7744_Partnerintro_events.csv",
    keep_event_columns = ["Time", "Cluster ID", "KSLabel"],
    time_interval_columns = ["Start (s)", "Stop (s)"],
    progress=True,
)

print(df.head(5))

In [ ]:
spike_df